## Reading data for England 

In [ ]:
import pandas as pd

df = pd.read_csv('../Extacted data/Pharm stats /Pharm_England_2021_2025.csv')
df.head(2)

In [ ]:
# have null value in column 'CHEMICAL_SUBSTANCE_BNF_DESCR'
df.info()

## Fill in nulls in CHEMICAL_SUBSTANCE_BNF_DESCR column from BNF_CHEMICAL_SUBSTANCE

In [ ]:
df['CHEMICAL_SUBSTANCE_BNF_DESCR'] = df['CHEMICAL_SUBSTANCE_BNF_DESCR'].fillna(df['BNF_CHEMICAL_SUBSTANCE'])

## Dropping columns 

In [ ]:
df = df.drop(columns=['REGIONAL_OFFICE_CODE','BNF_CHEMICAL_SUBSTANCE','TOTAL_QUANTITY','QUANTITY','ADQUSAGE'])

In [ ]:
df.head()

## Changing type and names of columns

In [ ]:
df['YEAR_MONTH'] = pd.to_datetime(df['YEAR_MONTH'], format='%Y%m').dt.strftime('%Y-%m')

In [ ]:
df = df.rename(columns={
    'YEAR_MONTH': 'date',
    'REGIONAL_OFFICE_NAME': 'region',
    'CHEMICAL_SUBSTANCE_BNF_DESCR': 'chemical_substance',
    'ITEMS': 'items',
    'NIC': 'net_cost ',
    'ACTUAL_COST': 'actual_cost'
})

In [ ]:
df.head()

## Filtering antidepressants from the list

In [ ]:
df['chemical_substance'].nunique()

In [ ]:
for item in sorted(df['chemical_substance'].unique()):
    print(item)

In [ ]:
antidepressants = [
    'Agomelatine', 'Amitriptyline hydrochloride', 'Amoxapine', 
    'Bupropion hydrochloride', 'Citalopram hydrobromide', 'Citalopram hydrochloride',
    'Clomipramine hydrochloride', 'Dosulepin hydrochloride', 'Doxepin',
    'Duloxetine hydrochloride', 'Escitalopram', 'Fluoxetine hydrochloride',
    'Flupentixol decanoate', 'Flupentixol hydrochloride', 'Fluvoxamine maleate',
    'Imipramine hydrochloride', 'Isocarboxazid', 'Lofepramine hydrochloride',
    'Mianserin hydrochloride', 'Mirtazapine', 'Moclobemide',
    'Nefazodone hydrochloride', 'Nortriptyline', 'Oxitriptan',
    'Paroxetine hydrochloride', 'Phenelzine sulfate', 'Reboxetine',
    'Sertraline hydrochloride', 'Tranylcypromine sulfate', 'Trazodone hydrochloride',
    'Trimipramine maleate', 'Tryptophan', 'Venlafaxine', 'Vortioxetine'
]

df_antidepressants = df[df['chemical_substance'].isin(antidepressants)]

print(df_antidepressants.shape)
print(df_antidepressants['chemical_substance'].nunique(), "unique substances")

In [ ]:
df_antidepressants.reset_index()

In [ ]:
# save to csv
df_antidepressants.to_csv("England_antidepressants_21_25.csv", index=False)

In [ ]:
# convert to monthly period
df_antidepressants["date"] = pd.PeriodIndex(df_antidepressants["date"], freq="M")

# filter range
df_filtered = df_antidepressants[
    (df["date"] >= "2021-01") &
    (df["date"] <= "2025-12")
]

# group and sum
result = (
    df_filtered
    .groupby(["date", "chemical_substance"])["items"]
    .sum()
    .reset_index()
)

print(result)

In [ ]:
# total items per month
monthly_total = (
    df_filtered
    .groupby("date")["items"]
    .sum()
    .div(1000)
    .reset_index()
)

print(monthly_total)